# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is built from the prepared content-level dataset created during the earlier data preparation work. I keep features that are available before the prediction point and remove identifiers, the target label, and fields that could leak the outcome.

Numeric features are filled with the median value so that missing values do not prevent modelling. Categorical features are filled with "Unknown" and then one-hot encoded. The resulting matrix contains only prediction-time features.

In [3]:
!git clone https://github.com/mkhlor006/Flyrank_internship_ML.git /content/Flyrank_internship_ML

Cloning into '/content/Flyrank_internship_ML'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 193 (delta 87), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 2.03 MiB | 7.61 MiB/s, done.
Resolving deltas: 100% (87/87), done.


In [4]:
from pathlib import Path

repo_path = Path("/content/Flyrank_internship_ML")

print("Repository exists:", repo_path.exists())
print("Repository contents:")
print(list(repo_path.iterdir())[:20])

Repository exists: True
Repository contents:
[PosixPath('/content/Flyrank_internship_ML/DATA_USE.md'), PosixPath('/content/Flyrank_internship_ML/GUIDE.md'), PosixPath('/content/Flyrank_internship_ML/02_your_first_readable_model.ipynb'), PosixPath('/content/Flyrank_internship_ML/outputs'), PosixPath('/content/Flyrank_internship_ML/data'), PosixPath('/content/Flyrank_internship_ML/.gitignore'), PosixPath('/content/Flyrank_internship_ML/skills'), PosixPath('/content/Flyrank_internship_ML/docs'), PosixPath('/content/Flyrank_internship_ML/scripts'), PosixPath('/content/Flyrank_internship_ML/.git'), PosixPath('/content/Flyrank_internship_ML/requirements.txt'), PosixPath('/content/Flyrank_internship_ML/AGENTS.md'), PosixPath('/content/Flyrank_internship_ML/notebooks'), PosixPath('/content/Flyrank_internship_ML/SETUP.md'), PosixPath('/content/Flyrank_internship_ML/README.md'), PosixPath('/content/Flyrank_internship_ML/CLAUDE.md'), PosixPath('/content/Flyrank_internship_ML/LICENSE'), PosixPath(

In [6]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nFiles/folders here:")
print(os.listdir(".")[:20])

repo_path = Path("/content/Flyrank_internship_ML")

print("\nRepository exists:", repo_path.exists())

if repo_path.exists():
    print("Repository contents:")
    print(os.listdir(repo_path)[:20])

Current working directory:
/content

Files/folders here:
['.config', 'Flyrank_internship_ML', 'sample_data']

Repository exists: True
Repository contents:
['DATA_USE.md', 'GUIDE.md', '02_your_first_readable_model.ipynb', 'outputs', 'data', '.gitignore', 'skills', 'docs', 'scripts', '.git', 'requirements.txt', 'AGENTS.md', 'notebooks', 'SETUP.md', 'README.md', 'CLAUDE.md', 'LICENSE', 'work', '01_first_look_and_discovery.ipynb', '.github']


In [7]:
%cd /content/Flyrank_internship_ML

from pathlib import Path

print("Current directory:", Path.cwd())

/content/Flyrank_internship_ML
Current directory: /content/Flyrank_internship_ML


In [9]:
from pathlib import Path

raw_path = Path("data/raw/content_refresh_anonymized.csv")

print("Current directory:", Path.cwd())
print("Raw dataset exists:", raw_path.exists())

if raw_path.exists():
    print("Raw dataset size:", raw_path.stat().st_size, "bytes")
else:
    print("Raw dataset NOT found.")
    print("\nContents of data/:")

    data_path = Path("data")
    if data_path.exists():
        for item in data_path.rglob("*"):
            print(item)
    else:
        print("data/ folder does not exist.")

Current directory: /content/Flyrank_internship_ML
Raw dataset exists: True
Raw dataset size: 6727670 bytes


In [10]:
from pathlib import Path
import pandas as pd

feature_path = Path("data/processed/refresh_feature_vector.csv")

print("Feature file exists:", feature_path.exists())

if feature_path.exists():
    df = pd.read_csv(feature_path)

    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("First 10 columns:")
    print(df.columns[:10].tolist())

Feature file exists: False


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# Load the prepared feature vector created during the earlier workflow.
feature_path = Path("data/processed/refresh_feature_vector.csv")

if not feature_path.exists():
    raise FileNotFoundError(
        f"Could not find {feature_path}. "
        "Make sure the repository is available in Colab and the processed feature file exists."
    )

df = pd.read_csv(feature_path)

print("Original shape:", df.shape)
print("Original columns:", len(df.columns))

# Numeric features used for prediction.
NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Categorical features used for prediction.
CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

TARGET = "is_declining_label"

required_columns = (
    ["content_id", "client_id", TARGET]
    + NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"These required columns are missing: {missing_columns}"
    )

# Keep the identifiers and target separately.
content_ids = df["content_id"].copy()
client_ids = df["client_id"].copy()
y = df[TARGET].copy()

# Prepare numeric features.
X_numeric = df[NUMERIC_FEATURES].copy()

for col in NUMERIC_FEATURES:
    X_numeric[col] = pd.to_numeric(X_numeric[col], errors="coerce")

numeric_missing_before = int(X_numeric.isna().sum().sum())

# Fill numeric missing values using the training-data-independent median
# available in this prepared feature vector.
for col in NUMERIC_FEATURES:
    X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())

# Prepare categorical features.
X_categorical = df[CATEGORICAL_FEATURES].copy()

categorical_missing_before = int(X_categorical.isna().sum().sum())

for col in CATEGORICAL_FEATURES:
    X_categorical[col] = X_categorical[col].astype("string").fillna("Unknown")

# One-hot encode categorical features.
X_categorical_encoded = pd.get_dummies(
    X_categorical,
    drop_first=False,
    dtype=int
)

# Combine numeric and categorical features.
X = pd.concat(
    [
        X_numeric.reset_index(drop=True),
        X_categorical_encoded.reset_index(drop=True),
    ],
    axis=1
)

# Final checks.
remaining_missing = int(X.isna().sum().sum())

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Encoded feature matrix shape:", X.shape)
print("Numeric missing values before fill:", numeric_missing_before)
print("Categorical missing values before fill:", categorical_missing_before)
print("Remaining missing values after fill:", remaining_missing)

print("\nTarget distribution:")
print(y.value_counts().sort_index())

assert remaining_missing == 0, "Missing values remain in the final feature matrix."

print("\nFeature vector built successfully.")

FileNotFoundError: Could not find data/processed/refresh_feature_vector.csv. Make sure the repository is available in Colab and the processed feature file exists.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.